# Fock-PARFLM d=384 Gamma Sweep with Geodesic Residual Analysis

## Purpose

Run a gamma sweep at `d=384` on OpenWebText using the **exact same
configuration** as the `e5c_plgate` training run, then compute the
**damped-geodesic residual** $\bar{R}(\gamma)$ on each sweep checkpoint.

This produces the dual-axis overlay:
- **PPL($\gamma$)** — which damping minimises perplexity?
- **$\bar{R}(\gamma)$** — which damping makes the dynamics most geodesic?

If $\arg\min_\gamma \text{PPL}(\gamma) \approx \arg\min_\gamma \bar{R}(\gamma)$,
we have a **mechanistic explanation** for why the architecture works.

## Configuration (matches e5c_plgate exactly)

| Parameter | Value |
|---|---|
| d | 384 |
| L | 16 |
| Registers M | 32 |
| V_theta | Depth-conditioned multi-context Gaussian (5 heads × 8 wells) |
| Xi | 5long (topk=16, dt=32, da=16, mh=4) |
| Embeddings | Untied (ob) |
| Reverse channel | Enabled (stable, QK-norm, soft-norm, pre-LN, per-layer gate) |
| Register repulsion | 0.05 |
| Schedule | WSD |
| Gamma candidates | [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50] |
| Sweep steps | 3,000 per candidate |

## Companion documents

- `companion_notes/Geodesic_Preservation_Experiment.md` — theory and design
- `companion_notes/Fock_Mechanism_Ablation_Study_d384_OpenWebText.md` — e5a vs e5c ablation
- `companion_notes/Fock-PARFLM_Scale-Up_Gamma_Sweep_Results_and_Damping_Regime_Analysis.md`


In [ ]:
# ── Cell 0: Sweep Configuration ────────────────────────────────────

GAMMA_CANDIDATES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
SWEEP_STEPS      = 3_000
SWEEP_EVAL_INTERVAL = 500

N_GEODESIC_BATCHES = 10
GEODESIC_SEED      = 42
GEODESIC_EPSILON   = 1e-6

# ── Architecture (matches e5c_plgate) ──────────────────────────────
D              = 384
L              = 16
N_REGISTERS    = 32
BLOCK_SIZE     = 512
VOCAB_SIZE     = 50257

V_THETA_VARIANT             = 'gaussian'
V_THETA_N_HEADS             = 5
V_THETA_WELLS_PER_HEAD      = 8
V_THETA_DEPTH_CONDITION     = True
V_THETA_DEPTH_CODE_INIT_STD = 0.02

XI_OVERRIDE    = '5long'
XI_ALPHA_INITS = [0.50, 0.75, 0.95, 0.99, 0.995]
XI_CHANNELS    = len(XI_ALPHA_INITS)

V_PHI_KIND     = 'structural_competitive'
V_PHI_N_HEADS  = 4
V_PHI_D_TYPE   = 32
V_PHI_D_ANGLE  = 16
TOP_K          = 16

REVERSE_CHANNEL              = True
REVERSE_CHANNEL_STABLE       = True
REVERSE_CHANNEL_PRE_LN       = True
REVERSE_CHANNEL_SOFT_NORM    = True
REVERSE_CHANNEL_WARMUP_STEPS = 4000
REVERSE_CHANNEL_PER_LAYER    = True

REGISTER_REPULSION       = True
REGISTER_REPULSION_COEFF = 0.05

USE_OUTPUT_BIAS = True
TIE_EMBEDDINGS  = False

LR             = 3e-4
LR_SCHEDULE    = 'wsd'
LAMBDA_V       = 1e-2
BATCH_SIZE     = 2
GRAD_ACCUM     = 8
GRAD_CLIP      = 1.0
GRAD_CLIP_VPHI = 0.3

# ── Causal probe intervals ──
CAUSAL_PROBE_INTERVAL       = 4000   # architectural probe every N steps (0 = off)
TRAINED_LEAK_PROBE_INTERVAL = 10000  # trained-scale probe + honest PPL every N steps (0 = off)
TRAINED_LEAK_PROBE_K        = 256    # honest-PPL target tokens
TRAINED_LEAK_PROBE_PAIRS    = 2      # future-perturbation window pairs

MAX_TRAIN_TOKENS = 1_000_000_000
VAL_TOKENS       = 2_000_000

print(f'Gamma sweep: {GAMMA_CANDIDATES}')
print(f'Steps per candidate: {SWEEP_STEPS:,}')
print(f'd={D}  L={L}  M={N_REGISTERS}  batch={BATCH_SIZE}×{GRAD_ACCUM} (eff={BATCH_SIZE*GRAD_ACCUM})')
print(f'Geodesic analysis: {N_GEODESIC_BATCHES} validation batches, seed={GEODESIC_SEED}')

In [ ]:
# ── Cell 1: Environment + Drive Mount ──────────────────────────────
import os, sys, gc, shutil, subprocess, json, time, math, copy
from pathlib import Path

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _sh('pip install -q transformers huggingface_hub pyarrow matplotlib')

    GDRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_fock_gamma_sweep_geodesic_d384')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)

    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    SWEEP_OUTPUT_DIR = GDRIVE_ROOT / 'gamma_sweep'
    RESULTS_DIR      = GDRIVE_ROOT / 'results'
    SWEEP_OUTPUT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    SWEEP_OUTPUT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / 'gamma_sweep_geodesic_d384'
    RESULTS_DIR = SWEEP_OUTPUT_DIR
    for d in [DATA_DIR, SWEEP_OUTPUT_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

print(f'DATA_DIR         = {DATA_DIR}')
print(f'SWEEP_OUTPUT_DIR = {SWEEP_OUTPUT_DIR}')
print(f'RESULTS_DIR      = {RESULTS_DIR}')

In [ ]:
# ── Cell 2: GPU Check + Imports ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print('WARNING: No GPU detected. This notebook requires CUDA.')

from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

print('Model imports OK')

In [ ]:
# ── Cell 3: Data Loading ───────────────────────────────────────────
from data_module import get_batch
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

# Try to reuse cached data from other experiments
for alt_name in [
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05',
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd',
    'semsimula_fock_structured_vtheta_owt_phase4',
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_fock_multicontext_vtheta_owt',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens ...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    CHUNK_SIZE = 50_000
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

In [ ]:
# ── Cell 4: Logfreq Surprisal ─────────────────────────────────────
LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_openwebtext.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_openwebtext.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    np.save(str(DRIVE_LOGFREQ), surprisal)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    print(f'  Computed logfreq surprisal -> {LOGFREQ_FILE}')

print(f'Logfreq: {LOGFREQ_FILE}')

In [ ]:
# ── Cell 5: Model Builder ─────────────────────────────────────────

def build_model(gamma, device=DEVICE):
    """Build a Fock-PARFLM model with the e5c_plgate configuration at the given gamma."""
    model_cfg = FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE,
        d=D,
        max_len=1024,
        L=L,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_FILE),
        init_gamma=1.0,
        fixed_gamma=gamma,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        fock_version='v2',
        n_registers=N_REGISTERS,
        reverse_channel=REVERSE_CHANNEL,
        reverse_channel_stable=REVERSE_CHANNEL_STABLE,
        reverse_channel_pre_ln=REVERSE_CHANNEL_PRE_LN,
        reverse_channel_soft_norm=REVERSE_CHANNEL_SOFT_NORM,
        reverse_channel_warmup_steps=REVERSE_CHANNEL_WARMUP_STEPS,
        reverse_channel_per_layer=REVERSE_CHANNEL_PER_LAYER,
        register_repulsion=REGISTER_REPULSION,
        register_repulsion_coeff=REGISTER_REPULSION_COEFF,
        # Prefix-causal register lifecycle (causal-leak fix; see
        # Fock-PARFLM_Causal_Leak_Audit_Results.md).  Must be True for any
        # trustworthy sweep.
        prefix_causal_registers=True,
        v_phi_kind=V_PHI_KIND,
        v_phi_n_heads=V_PHI_N_HEADS,
        v_phi_d_type=V_PHI_D_TYPE,
        v_phi_d_angle=V_PHI_D_ANGLE,
        top_k=TOP_K,
        use_output_bias=USE_OUTPUT_BIAS,
        tie_embeddings=TIE_EMBEDDINGS,
    )
    model = FockMultiXiPARFLM(model_cfg).to(device)

    if V_THETA_VARIANT == 'gaussian':
        from model_gaussian_vtheta import (
            DepthConditionedMultiContextGaussianVTheta,
            install_depth_routing,
        )
        model.V_theta = DepthConditionedMultiContextGaussianVTheta(
            d=D,
            K=V_THETA_WELLS_PER_HEAD,
            n_ctx=V_THETA_N_HEADS,
            n_layers=L,
            code_init_std=V_THETA_DEPTH_CODE_INIT_STD,
        ).to(device)
        install_depth_routing(model)

    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Model built: gamma={gamma:.3f}  params={n_params:,}')
    return model, model_cfg


# Quick sanity check
_test_model, _test_cfg = build_model(0.30)
del _test_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Model builder OK')

In [ ]:
# ── Cell 6: Training + Evaluation Helpers ──────────────────────────

def forward_fock_with_vreg(model, x, targets):
    """Forward pass returning (loss, ntp_loss, v_reg)."""
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.float().reshape(-1, VOCAB_SIZE),
        targets.reshape(-1),
    )
    v_reg = torch.tensor(0.0, device=x.device)
    if LAMBDA_V > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_reg = (V_vals.float() ** 2).mean()
        loss = loss_ntp + LAMBDA_V * v_reg
    else:
        loss = loss_ntp
    return loss, loss_ntp, v_reg


@torch.no_grad()
def evaluate(model, val_ids, n_iters=40):
    """Evaluate on validation set. Returns (val_loss, val_ppl)."""
    model.eval()
    rng = np.random.default_rng(42)
    losses = []
    for _ in range(n_iters):
        x_np, y_np = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(x_np).to(DEVICE)
        y = torch.from_numpy(y_np).to(DEVICE)
        with torch.enable_grad():
            _, ntp, _ = forward_fock_with_vreg(model, x, y)
        losses.append(ntp.item())
    model.train()
    val_loss = sum(losses) / len(losses)
    val_ppl = math.exp(val_loss)
    return val_loss, val_ppl


def get_wsd_lr(step, total_steps, peak_lr, floor_lr=None):
    """WSD learning rate schedule."""
    if floor_lr is None:
        floor_lr = peak_lr * 0.05
    warmup_steps = int(total_steps * 0.05)
    stable_end   = int(total_steps * 0.65)
    if step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    elif step < stable_end:
        return peak_lr
    else:
        decay_steps = total_steps - stable_end
        progress = (step - stable_end) / max(decay_steps, 1)
        return floor_lr + 0.5 * (peak_lr - floor_lr) * (1 + math.cos(math.pi * progress))


print('Training helpers OK')

In [ ]:
# ── Cell 7: Per-Group Gradient Clipping ────────────────────────────

CLIP_OVERRIDES = {
    'V_phi': 0.3,
    'creation_gate': 0.3,
    'destruction_gate': 0.3,
    'reverse_channel_scale': 0.1,
    'reverse_ch': 0.1,
    'register': 0.3,
    'depth_code': 0.5,
}

REVERSE_CH_EXCLUDE = {'override:reverse_ch', 'override:reverse_channel_scale'}


def assign_param_group(name):
    """Assign a parameter to its clip group."""
    for key in CLIP_OVERRIDES:
        if key in name:
            return f'override:{key}'
    if 'embed' in name and 'pos' not in name:
        return 'E'
    if 'pos' in name:
        return 'P'
    if 'V_theta' in name or 'v_theta' in name:
        return 'V_theta'
    if 'score' in name or 'lm_head' in name:
        return 'score_head'
    return 'other'


def clip_grad_per_group(model):
    """Apply per-group gradient clipping. Returns (total_norm, top_group, top_norm)."""
    groups = {}
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        g = assign_param_group(name)
        groups.setdefault(g, []).append(p)

    group_norms = {}
    for g, params in groups.items():
        clip_val = CLIP_OVERRIDES.get(g.replace('override:', ''), GRAD_CLIP)
        nn.utils.clip_grad_norm_(params, clip_val)
        norm = sum(p.grad.data.norm().item() ** 2 for p in params) ** 0.5
        group_norms[g] = norm

    all_params = [p for p in model.parameters() if p.grad is not None]
    total_norm = sum(p.grad.data.norm().item() ** 2 for p in all_params) ** 0.5

    filtered = {k: v for k, v in group_norms.items() if k not in REVERSE_CH_EXCLUDE}
    top_group = max(filtered, key=filtered.get) if filtered else 'none'
    top_norm = filtered.get(top_group, 0.0)

    return total_norm, top_group, top_norm


print('Per-group clipping OK')

In [ ]:
# ── Cell 8: Geodesic Residual Functions ────────────────────────────
# Inlined from geodesic_residual.py for Colab self-containment.

from typing import Dict, List, Tuple


def collect_trajectory(model, x):
    """Forward pass returning per-layer hidden states [h_0, ..., h_L]."""
    with torch.enable_grad():
        h0 = model._embed(x)
        _, traj = model._stack_forward(h0, x, return_trajectory=True)
    return traj


@torch.no_grad()
def vtheta_value_and_grad(model, h_ell, layer_idx):
    """Evaluate V_theta and analytical gradient at a given layer. Returns (V, grad_V)."""
    xis = model.xi_module(h_ell)
    model.V_theta.set_active_layer(layer_idx)
    V = model.V_theta(xis, h_ell).squeeze(-1)
    grad_V = model.V_theta.analytical_grad(xis, h_ell)
    return V, grad_V


def conformal_grad(grad_V, E_minus_V, epsilon=1e-6):
    """Gradient of the Jacobi conformal factor phi = 0.5 * log(2*(E-V))."""
    denom = 2.0 * E_minus_V.unsqueeze(-1).clamp(min=epsilon)
    return -grad_V / denom


def christoffel_vv(phi_grad, v):
    """Christoffel-velocity contraction: Gamma^k_ij v^i v^j for conformally flat metric."""
    phi_dot_v = (phi_grad * v).sum(dim=-1, keepdim=True)
    v_sq = (v * v).sum(dim=-1, keepdim=True)
    return 2.0 * phi_dot_v * v - v_sq * phi_grad


@torch.no_grad()
def compute_residual(traj, model, gamma_eval, device, epsilon=1e-6, ref_layer=0):
    """Compute the damped-geodesic residual R_bar for one trajectory.
    
    Returns dict: R_bar, per_layer_R, excluded_frac, gamma_geo.
    """
    L = len(traj) - 1

    h_ref = traj[ref_layer].to(device)
    v_ref = traj[ref_layer + 1].to(device) - h_ref
    KE_ref = 0.5 * (v_ref * v_ref).sum(dim=-1)
    V_ref, _ = vtheta_value_and_grad(model, h_ref, ref_layer)
    E = KE_ref + V_ref
    del h_ref, v_ref, KE_ref, V_ref

    per_layer_R = []
    total_excluded = 0
    total_tokens = 0
    num_sum = 0.0
    den_sum = 0.0

    for ell in range(1, L):
        h_prev = traj[ell - 1].to(device)
        h_curr = traj[ell].to(device)
        h_next = traj[ell + 1].to(device)

        v_ell = h_curr - h_prev
        v_next = h_next - h_curr
        a_ell = v_next - v_ell

        V_ell, grad_V_ell = vtheta_value_and_grad(model, h_curr, ell)
        E_minus_V = E - V_ell
        allowed = E_minus_V > epsilon
        total_excluded += int((~allowed).sum().item())
        total_tokens += allowed.numel()

        phi_g = conformal_grad(grad_V_ell, E_minus_V, epsilon)
        Gamma_vv = christoffel_vv(phi_g, v_ell)

        residual_vec = a_ell + Gamma_vv + gamma_eval * v_ell
        residual_norm = residual_vec.norm(dim=-1)
        a_norm = a_ell.norm(dim=-1)
        R_ell = residual_norm / (a_norm + epsilon)

        R_ell = R_ell * allowed.float()
        n_allowed = allowed.float().sum().clamp(min=1.0)
        mean_R = R_ell.sum() / n_allowed
        per_layer_R.append(float(mean_R.item()))

        a_plus_Gamma = a_ell + Gamma_vv
        dot_num = (a_plus_Gamma * v_ell).sum(dim=-1)
        v_sq = (v_ell * v_ell).sum(dim=-1)
        num_sum += float((dot_num * allowed.float()).sum().item())
        den_sum += float((v_sq * allowed.float()).sum().item())

        del h_prev, h_curr, h_next, v_ell, v_next, a_ell
        del V_ell, grad_V_ell, phi_g, Gamma_vv

    R_bar = float(np.mean(per_layer_R)) if per_layer_R else float('inf')
    excluded_frac = total_excluded / max(total_tokens, 1)
    gamma_geo = -num_sum / max(den_sum, 1e-12)

    return {
        'R_bar': R_bar,
        'per_layer_R': per_layer_R,
        'excluded_frac': excluded_frac,
        'gamma_geo': gamma_geo,
    }


print('Geodesic residual functions OK')

In [ ]:
# ── Cell 9b: Two-Stage Causal Probes ─────────────────────────────
# Stage 1 — lightweight architectural probe (CPU, float64, tiny model)
# Stage 2 — trained-scale leak probe + honest PPL (live trained model)

def run_causal_probe(step_num, model_cfg_ref):
    """Lightweight causal-integrity probe (runs on CPU in float64).

    Builds a tiny model with the same structural features as the training
    config, opens the reverse channel fully, and checks that perturbing
    future tokens produces exactly zero change in logits at earlier
    positions.  Returns (passed: bool, max_delta: float).
    """
    import math as _math
    _PROBE_VOCAB, _PROBE_D, _PROBE_L = 101, 32, 4
    _PROBE_T, _PROBE_M, _PROBE_XI = 48, 8, 3
    _PROBE_WELLS = 4

    _logfreq_probe = Path('/tmp/causal_probe_logfreq.npy')
    np.save(_logfreq_probe, np.full(_PROBE_VOCAB, 5.0, dtype=np.float32))

    _probe_cfg = FockMultiXiPARFConfig(
        vocab_size=_PROBE_VOCAB, d=_PROBE_D, max_len=64, L=_PROBE_L,
        v_hidden=64, v_depth=3, dt=1.0,
        mass_mode='logfreq', logfreq_path=str(_logfreq_probe),
        logfreq_init_alpha=0.1, init_gamma=1.0, fixed_gamma=0.30,
        causal_force=True, ln_after_step=True,
        xi_channels=_PROBE_XI, xi_alpha_inits=[0.5, 0.9, 0.99],
        xi_learnable=True, xi_alpha_init_mode='explicit',
        v_phi_kind='structural_competitive',
        v_phi_d_type=8, v_phi_d_angle=4, v_phi_eps=0.1,
        v_phi_phi_hidden=16, v_phi_theta_hidden=16, v_phi_mlp_hidden=16,
        top_k=8, v_phi_n_heads=2,
        use_output_bias=True, tie_embeddings=False,
        score_head_hidden=8,
        gumbel_tau_init=1.0, gumbel_tau_min=0.3, gumbel_noise=True,
        use_gathered_v_phi=True, use_layer_checkpoint=False,
        ln_before_distance=True, per_layer_v_phi_scale=True,
        fock_version='v2', n_registers=_PROBE_M,
        register_salience_decay=0.5, register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True,
        d_k=16, tau_create_init=8.0,
        reverse_channel=True, reverse_channel_stable=True,
        reverse_channel_pre_ln=True, reverse_channel_soft_norm=True,
        reverse_channel_warmup_steps=4000, reverse_channel_per_layer=True,
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True, register_repulsion=False,
        prefix_causal_registers=True,
    )
    torch.manual_seed(1234)
    _probe_model = FockMultiXiPARFLM(_probe_cfg)
    if V_THETA_VARIANT == 'gaussian':
        from model_gaussian_vtheta import (
            DepthConditionedMultiContextGaussianVTheta as _DCMCGVT,
            install_depth_routing as _idr,
        )
        _probe_model.V_theta = _DCMCGVT(
            d=_PROBE_D, K=_PROBE_WELLS, n_ctx=_PROBE_XI, n_layers=_PROBE_L,
            w_scale=1.0, init_log_precision=-_math.log(_PROBE_D),
            precision_max=2.0/_PROBE_D, code_init_std=0.02,
        )
        _idr(_probe_model)
    _probe_model.double().eval()

    with torch.no_grad():
        _probe_model.reverse_channel_scale.fill_(1.0)
        _probe_model.reverse_warmup_step.fill_(4000)

    _t_p = _PROBE_T // 2
    _prng = np.random.default_rng(7)
    _x1 = torch.from_numpy(_prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T))).long()
    _x2 = _x1.clone()
    _x2[:, _t_p:] = torch.from_numpy(
        _prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T - _t_p))).long()

    with torch.enable_grad():
        _la = _probe_model(_x1)[0].detach()
        _lb = _probe_model(_x2)[0].detach()
    _max_delta = float((_la[:, :_t_p] - _lb[:, :_t_p]).abs().max().item())

    _probe_model.train()
    torch.manual_seed(99)
    with torch.enable_grad():
        _lta = _probe_model(_x1)[0].detach()
    torch.manual_seed(99)
    with torch.enable_grad():
        _ltb = _probe_model(_x2)[0].detach()
    _probe_model.eval()
    _max_delta_train = float((_lta[:, :_t_p] - _ltb[:, :_t_p]).abs().max().item())

    _max_delta = max(_max_delta, _max_delta_train)
    _passed = (_max_delta == 0.0)

    del _probe_model, _la, _lb, _lta, _ltb, _x1, _x2
    gc.collect()

    status = 'PASS' if _passed else '*** FAIL ***'
    print(f'\n[causal probe] step {step_num:,}  max|dlogit|={_max_delta:.3e}  [{status}]')
    if not _passed:
        print('[causal probe] WARNING: nonzero future sensitivity detected!')
        print('[causal probe] The prefix_causal_registers fix may not be working correctly.')
    return _passed, _max_delta


def run_trained_leak_probe(step_num, model, val_ids_arr):
    """Run the trained-scale leak probe (Step 11) on the live model.

    Uses the actual trained weights and validation data to measure:
      Part 1: future-perturbation effect on past logits/NLL (float32, GPU)
      Part 2: honest PPL vs standard PPL (leak-free scoring)
    Returns a dict with probe and honest-PPL results.
    """
    _debug_dir = str(CA_DIR / 'scaleup' / 'debug')
    if _debug_dir not in sys.path:
        sys.path.insert(0, _debug_dir)
    from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

    print(f'\n{"="*64}')
    print(f'[trained leak probe] step {step_num:,} — running on live model')
    print(f'{"="*64}')

    probe_res = probe_trained_leak(
        model, val_ids_arr, device=DEVICE, context=BLOCK_SIZE,
        n_pairs=TRAINED_LEAK_PROBE_PAIRS, use_float64=False)

    honest_res = honest_ppl_test(
        model, val_ids_arr, k=TRAINED_LEAK_PROBE_K,
        context=BLOCK_SIZE, batch=BATCH_SIZE, device=DEVICE)

    model.train()

    result = {
        'step': step_num,
        'event': 'trained_leak_probe',
        'probe_max_dlogit_past': probe_res['max_dlogit_past'],
        'probe_mean_dnll_past_nats': round(probe_res['mean_dnll_past'], 6),
        'probe_gate_zero_control': probe_res['gate_zero_control'],
        'honest_k': honest_res['k'],
        'ppl_mid_window_standard': round(honest_res['ppl_mid_window'], 4),
        'ppl_last_pos_leak_free': round(honest_res['ppl_last_pos'], 4),
        'paired_diff_nats': round(honest_res['paired_diff_nats'], 6),
        'paired_diff_se': round(honest_res['paired_diff_se'], 6),
    }

    _leak_status = 'CLEAN' if result['paired_diff_nats'] < 0.1 else 'LEAK DETECTED'
    print(f'\n[trained leak probe] step {step_num:,}  '
          f'honest_PPL={result["ppl_last_pos_leak_free"]:.2f}  '
          f'standard_PPL={result["ppl_mid_window_standard"]:.2f}  '
          f'diff={result["paired_diff_nats"]:+.4f} nats  [{_leak_status}]')
    return result


print('Two-stage causal probes OK')

In [ ]:
# ── Cell 9: Single-Gamma Training Function ─────────────────────────

def train_one_gamma(gamma, sweep_dir, train_ids, val_ids):
    """Train one gamma candidate and save checkpoint. Returns result dict.

    Resume-aware: if ckpt_best.pt exists but its stored step < SWEEP_STEPS,
    the run is treated as incomplete — model + optimizer are restored and
    training continues from the next step.  Only a checkpoint whose step
    >= SWEEP_STEPS is considered truly finished.
    """
    gamma_dir = sweep_dir / f'gamma_{gamma:.3f}'
    ckpt_dir  = gamma_dir / 'checkpoints'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    log_path  = gamma_dir / 'training_log.jsonl'

    best_ckpt = ckpt_dir / 'ckpt_best.pt'
    resume_step = 0
    resume_data = None

    if best_ckpt.exists():
        ckpt = torch.load(str(best_ckpt), map_location='cpu', weights_only=False)
        ckpt_step = ckpt.get('step', 0)
        best_ppl_loaded = ckpt.get('val_ppl', float('inf'))

        if ckpt_step >= SWEEP_STEPS:
            print(f'  [SKIP] gamma={gamma:.3f} already complete '
                  f'(step {ckpt_step:,}/{SWEEP_STEPS:,}) — best PPL={best_ppl_loaded:.2f}')
            del ckpt
            return {
                'gamma': gamma,
                'best_ppl': best_ppl_loaded,
                'skipped': True,
            }
        else:
            print(f'  [RESUME] gamma={gamma:.3f} incomplete '
                  f'(step {ckpt_step:,}/{SWEEP_STEPS:,}, PPL={best_ppl_loaded:.2f}) '
                  f'— continuing for {SWEEP_STEPS - ckpt_step:,} more steps')
            resume_step = ckpt_step
            resume_data = ckpt

    print(f'\n{"="*60}')
    print(f'  GAMMA = {gamma:.3f}  ({SWEEP_STEPS:,} steps)')
    print(f'{"="*60}')

    model, model_cfg = build_model(gamma)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.1)

    best_ppl = float('inf')
    best_loss = float('inf')

    if resume_data is not None:
        model.load_state_dict(resume_data['model_state_dict'], strict=False)
        if 'optimizer_state_dict' in resume_data:
            try:
                optimizer.load_state_dict(resume_data['optimizer_state_dict'])
                print(f'  Optimizer state restored.')
            except (ValueError, KeyError) as e:
                print(f'  [info] Optimizer state incompatible, starting fresh: {e}')
        best_ppl = resume_data.get('val_ppl', float('inf'))
        best_loss = resume_data.get('val_loss', float('inf'))
        del resume_data

    model.train()

    rng = np.random.default_rng(0)
    if resume_step > 0:
        for _ in range(resume_step * GRAD_ACCUM):
            get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)

    t0 = time.time()

    log_entries = []
    if log_path.exists():
        with open(str(log_path)) as f:
            for line in f:
                try:
                    log_entries.append(json.loads(line))
                except json.JSONDecodeError:
                    pass

    for step in range(resume_step + 1, SWEEP_STEPS + 1):
        lr = get_wsd_lr(step, SWEEP_STEPS, LR)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        model.train()
        optimizer.zero_grad(set_to_none=True)

        accum_ntp = 0.0
        accum_vreg = 0.0
        for _micro in range(GRAD_ACCUM):
            x_np, y_np = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
            x = torch.from_numpy(x_np).to(DEVICE)
            y = torch.from_numpy(y_np).to(DEVICE)
            loss, ntp, vreg = forward_fock_with_vreg(model, x, y)
            if REGISTER_REPULSION:
                _rep = model.pop_repulsion_loss()
                loss = loss + _rep
            (loss / GRAD_ACCUM).backward()
            accum_ntp += ntp.item() / GRAD_ACCUM
            accum_vreg += float(vreg.detach()) / GRAD_ACCUM
            del loss, ntp, vreg, x, y

        total_norm, top_group, top_norm = clip_grad_per_group(model)
        optimizer.step()

        if step % 50 == 0:
            elapsed = time.time() - t0
            steps_done = step - resume_step
            remaining = elapsed / steps_done * (SWEEP_STEPS - step)
            print(f'  step {step:>5d}/{SWEEP_STEPS}  ntp={accum_ntp:.4f}  '
                  f'v_reg={accum_vreg:.4f}  lr={lr:.2e}  grad={total_norm:.2f}  '
                  f'top[{top_group}]={top_norm:.1f}  '
                  f'{elapsed:.0f}s (~{remaining/3600:.1f}h remaining)')

        if step % SWEEP_EVAL_INTERVAL == 0 or step == SWEEP_STEPS:
            val_loss, val_ppl = evaluate(model, val_ids)
            is_best = val_ppl < best_ppl
            if is_best:
                best_ppl = val_ppl
                best_loss = val_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'step': step,
                'val_loss': best_loss,
                'val_ppl': best_ppl,
                'gamma': gamma,
            }, str(best_ckpt))
            marker = ' *** NEW BEST ***' if is_best else ''
            print(f'  >>> EVAL step {step:,}  val_loss={val_loss:.4f}  '
                  f'val_ppl={val_ppl:.2f}  best={best_ppl:.2f}{marker}')
            log_entries.append({
                'step': step, 'val_loss': val_loss, 'val_ppl': val_ppl,
                'best_ppl': best_ppl, 'train_ntp': accum_ntp,
            })

        # Stage 1: architectural causal probe
        if CAUSAL_PROBE_INTERVAL > 0 and step % CAUSAL_PROBE_INTERVAL == 0:
            _cp_passed, _cp_delta = run_causal_probe(step, model_cfg)
            log_entries.append({
                'step': step, 'event': 'causal_probe',
                'causal_probe_passed': _cp_passed,
                'causal_probe_max_delta': _cp_delta,
            })

        # Stage 2: trained-scale leak probe + honest PPL
        if TRAINED_LEAK_PROBE_INTERVAL > 0 and step % TRAINED_LEAK_PROBE_INTERVAL == 0:
            _tlp = run_trained_leak_probe(step, model, val_ids)
            log_entries.append(_tlp)

    elapsed = time.time() - t0
    print(f'  Done gamma={gamma:.3f}  best_ppl={best_ppl:.2f}  ({elapsed:.0f}s)')

    with open(str(log_path), 'w') as f:
        for entry in log_entries:
            f.write(json.dumps(entry) + '\n')

    del model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'gamma': gamma,
        'best_ppl': best_ppl,
        'best_loss': best_loss,
        'wall_clock_s': elapsed,
        'skipped': False,
    }


print('Training function OK')

In [ ]:
# ── Cell 10: Run Gamma Sweep ───────────────────────────────────────

sweep_results = []
t_sweep_start = time.time()

print(f'Starting gamma sweep: {len(GAMMA_CANDIDATES)} candidates × {SWEEP_STEPS:,} steps')
print(f'Candidates: {GAMMA_CANDIDATES}')
print()

for gi, gamma in enumerate(GAMMA_CANDIDATES):
    print(f'\n[{gi+1}/{len(GAMMA_CANDIDATES)}] gamma={gamma:.3f}')
    result = train_one_gamma(gamma, SWEEP_OUTPUT_DIR, train_ids, val_ids)
    sweep_results.append(result)

sweep_results.sort(key=lambda r: r['best_ppl'])

t_sweep_elapsed = time.time() - t_sweep_start
print(f'\n{"="*60}')
print(f'  GAMMA SWEEP COMPLETE  ({t_sweep_elapsed/3600:.1f}h total)')
print(f'{"="*60}')
print(f'{"Rank":>4s}  {"gamma":>8s}  {"Best PPL":>10s}  {"Wall (s)":>10s}')
for i, r in enumerate(sweep_results):
    skip = ' [cached]' if r.get('skipped') else ''
    wc = r.get('wall_clock_s', 0)
    print(f'{i+1:>4d}  {r["gamma"]:8.3f}  {r["best_ppl"]:10.2f}  {wc:10.0f}{skip}')

summary = {
    'preset': 'sweep-d384-e5c_plgate',
    'd': D, 'L': L,
    'sweep_steps': SWEEP_STEPS,
    'results': sweep_results,
}
summary_path = SWEEP_OUTPUT_DIR / 'sweep_summary.json'
with open(str(summary_path), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSummary saved: {summary_path}')

In [ ]:
# ── Cell 11: Geodesic Residual Analysis on Sweep Checkpoints ──────

print(f'\n{"="*60}')
print(f'  GEODESIC RESIDUAL ANALYSIS')
print(f'  d={D}  L={L}  batches={N_GEODESIC_BATCHES}  seed={GEODESIC_SEED}')
print(f'{"="*60}\n')

# Prepare fixed validation batches (same for ALL checkpoints)
geo_rng = np.random.default_rng(GEODESIC_SEED)
val_batches = []
for _ in range(N_GEODESIC_BATCHES):
    xb, _ = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, geo_rng)
    val_batches.append(torch.from_numpy(xb))

geodesic_results = []

for gi, gamma in enumerate(GAMMA_CANDIDATES):
    ckpt_path = SWEEP_OUTPUT_DIR / f'gamma_{gamma:.3f}' / 'checkpoints' / 'ckpt_best.pt'
    if not ckpt_path.exists():
        print(f'  [SKIP] gamma={gamma:.3f} — no checkpoint found')
        continue

    t0 = time.time()
    print(f'--- [{gi+1}/{len(GAMMA_CANDIDATES)}] gamma={gamma:.3f} ---')

    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    gamma_train = ckpt.get('gamma', gamma)
    val_ppl = ckpt.get('val_ppl', float('inf'))

    model, model_cfg = build_model(gamma_train)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    model.eval()
    del ckpt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    batch_results = []
    for bi, x_batch in enumerate(val_batches):
        x = x_batch.to(DEVICE)
        traj = collect_trajectory(model, x)
        res = compute_residual(traj, model, gamma_train, DEVICE, epsilon=GEODESIC_EPSILON)
        batch_results.append(res)
        del traj

    R_bar = float(np.mean([r['R_bar'] for r in batch_results]))
    gamma_geo = float(np.mean([r['gamma_geo'] for r in batch_results]))
    excluded_frac = float(np.mean([r['excluded_frac'] for r in batch_results]))
    per_layer_R = [
        float(np.mean([br['per_layer_R'][i] for br in batch_results]))
        for i in range(len(batch_results[0]['per_layer_R']))
    ]

    entry = {
        'gamma_train': gamma_train,
        'val_ppl': val_ppl,
        'R_bar': R_bar,
        'gamma_geo': gamma_geo,
        'excluded_frac': excluded_frac,
        'per_layer_R': per_layer_R,
    }
    geodesic_results.append(entry)

    elapsed = time.time() - t0
    print(f'  R_bar={R_bar:.4f}  gamma_geo={gamma_geo:.4f}  '
          f'excluded={excluded_frac:.4f}  PPL={val_ppl:.2f}  ({elapsed:.1f}s)')

    del model, model_cfg
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Summary table
geodesic_results.sort(key=lambda r: r['gamma_train'])
print(f'\n{"="*60}')
print(f'  GEODESIC RESIDUAL RESULTS  d={D}  L={L}')
print(f'{"="*60}')
print(f'{"gamma":>8s}  {"PPL":>8s}  {"R_bar":>8s}  {"gamma_geo":>10s}  {"excl%":>6s}')
print(f'{"-----":>8s}  {"---":>8s}  {"-----":>8s}  {"--------":>10s}  {"-----":>6s}')

best_r_idx = int(np.argmin([r['R_bar'] for r in geodesic_results]))
best_ppl_idx = int(np.argmin([r['val_ppl'] for r in geodesic_results]))

for i, r in enumerate(geodesic_results):
    markers = []
    if i == best_r_idx:
        markers.append('R*')
    if i == best_ppl_idx:
        markers.append('PPL*')
    marker = '  <-- ' + ', '.join(markers) if markers else ''
    print(f'{r["gamma_train"]:8.3f}  {r["val_ppl"]:8.2f}  '
          f'{r["R_bar"]:8.4f}  {r["gamma_geo"]:10.4f}  '
          f'{r["excluded_frac"]*100:5.1f}%{marker}')

coincidence = (geodesic_results[best_r_idx]['gamma_train']
               == geodesic_results[best_ppl_idx]['gamma_train'])
if coincidence:
    print(f'\n  >>> MINIMA COINCIDE at gamma={geodesic_results[best_r_idx]["gamma_train"]:.3f}')
    print(f'      Mechanistic claim supported: the damping that minimises')
    print(f'      PPL also minimises the geodesic residual.')
else:
    print(f'\n  >>> PPL minimum at gamma={geodesic_results[best_ppl_idx]["gamma_train"]:.3f}, '
          f'R_bar minimum at gamma={geodesic_results[best_r_idx]["gamma_train"]:.3f}')
    print(f'      Gap = {abs(geodesic_results[best_ppl_idx]["gamma_train"] - geodesic_results[best_r_idx]["gamma_train"]):.3f}')

# Save full results
geo_json_path = RESULTS_DIR / 'geodesic_results.json'
with open(str(geo_json_path), 'w') as f:
    json.dump({
        'd': D, 'L': L,
        'n_batches': N_GEODESIC_BATCHES,
        'seed': GEODESIC_SEED,
        'epsilon': GEODESIC_EPSILON,
        'results': geodesic_results,
    }, f, indent=2)
print(f'\nResults saved: {geo_json_path}')

In [ ]:
# ── Cell 12: Overlay Plot — PPL(gamma) vs R_bar(gamma) ─────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

gammas = [r['gamma_train'] for r in geodesic_results]
ppls   = [r['val_ppl'] for r in geodesic_results]
r_bars = [r['R_bar'] for r in geodesic_results]
g_geos = [r['gamma_geo'] for r in geodesic_results]

# ── Figure 1: Dual-axis overlay ──
fig, ax1 = plt.subplots(figsize=(10, 6))

color_ppl = '#2563eb'
color_r = '#dc2626'

ax1.set_xlabel(r'$\gamma_{\mathrm{train}}$', fontsize=13)
ax1.set_ylabel('Perplexity (PPL)', color=color_ppl, fontsize=13)
ax1.plot(gammas, ppls, 'o-', color=color_ppl, linewidth=2, markersize=7, label='PPL')
ax1.tick_params(axis='y', labelcolor=color_ppl)

ax2 = ax1.twinx()
ax2.set_ylabel(r'$\bar{R}(\gamma)$  (geodesic residual)', color=color_r, fontsize=13)
ax2.plot(gammas, r_bars, 's--', color=color_r, linewidth=2, markersize=7, label=r'$\bar{R}$')
ax2.tick_params(axis='y', labelcolor=color_r)

best_ppl_g = gammas[int(np.argmin(ppls))]
best_r_g = gammas[int(np.argmin(r_bars))]
ax1.axvline(best_ppl_g, color=color_ppl, linestyle=':', alpha=0.5,
            label=f'PPL min @ {best_ppl_g:.3f}')
ax2.axvline(best_r_g, color=color_r, linestyle=':', alpha=0.5,
            label=rf'$\bar{{R}}$ min @ {best_r_g:.3f}')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10)

ax1.set_title(f'd={D}  L={L}  |  PPL vs Geodesic Residual $\\bar{{R}}(\\gamma)$',
              fontsize=14, pad=12)
fig.tight_layout()

overlay_path = RESULTS_DIR / 'geodesic_overlay_d384.png'
fig.savefig(str(overlay_path), dpi=150, bbox_inches='tight')
print(f'Overlay figure saved: {overlay_path}')
plt.show()

# ── Figure 2: gamma_geo recovery ──
fig2, ax3 = plt.subplots(figsize=(8, 5))
ax3.plot(gammas, g_geos, 'D-', color='#059669', linewidth=2, markersize=8)
ax3.axhline(best_ppl_g, color=color_ppl, linestyle='--', alpha=0.5,
            label=f'PPL-optimal gamma = {best_ppl_g:.3f}')
ax3.set_xlabel(r'$\gamma_{\mathrm{train}}$', fontsize=13)
ax3.set_ylabel(r'$\gamma_{\mathrm{geo}}$ (recovered intrinsic damping)', fontsize=13)
ax3.set_title(f'd={D}  L={L}  |  Recovered Intrinsic Damping', fontsize=14, pad=12)
ax3.legend(fontsize=10)
fig2.tight_layout()

geo_recovery_path = RESULTS_DIR / 'gamma_geo_recovery_d384.png'
fig2.savefig(str(geo_recovery_path), dpi=150, bbox_inches='tight')
print(f'Gamma_geo recovery figure saved: {geo_recovery_path}')
plt.show()

# ── Figure 3: Per-layer residual heatmap ──
fig3, ax4 = plt.subplots(figsize=(10, 6))
layer_matrix = np.array([r['per_layer_R'] for r in geodesic_results])
im = ax4.imshow(layer_matrix, aspect='auto', origin='lower', cmap='viridis')
ax4.set_yticks(range(len(gammas)))
ax4.set_yticklabels([f'{g:.2f}' for g in gammas])
ax4.set_xlabel('Layer index', fontsize=13)
ax4.set_ylabel(r'$\gamma_{\mathrm{train}}$', fontsize=13)
ax4.set_title(f'd={D}  L={L}  |  Per-Layer Geodesic Residual', fontsize=14, pad=12)
fig3.colorbar(im, ax=ax4, label=r'$R_\ell$')
fig3.tight_layout()

perlayer_path = RESULTS_DIR / 'geodesic_per_layer_d384.png'
fig3.savefig(str(perlayer_path), dpi=150, bbox_inches='tight')
print(f'Per-layer heatmap saved: {perlayer_path}')
plt.show()

In [ ]:
# ── Cell 13: Final Summary ─────────────────────────────────────────

print(f'\n{"="*60}')
print(f'  d={D} GAMMA SWEEP + GEODESIC RESIDUAL — FINAL SUMMARY')
print(f'{"="*60}')
print(f'  PPL-optimal gamma:       {geodesic_results[best_ppl_idx]["gamma_train"]:.3f} '
      f'(PPL={geodesic_results[best_ppl_idx]["val_ppl"]:.2f})')
print(f'  R_bar-optimal gamma:     {geodesic_results[best_r_idx]["gamma_train"]:.3f} '
      f'(R_bar={geodesic_results[best_r_idx]["R_bar"]:.4f})')
if coincidence:
    print(f'  >>> COINCIDENCE: PPL and R_bar minima align at '
          f'gamma={geodesic_results[best_r_idx]["gamma_train"]:.3f}')
    print(f'  >>> The damping that minimises perplexity is the one')
    print(f'      that makes the dynamics most geodesic.')
else:
    print(f'  >>> NON-COINCIDENCE: gap = '
          f'{abs(geodesic_results[best_ppl_idx]["gamma_train"] - geodesic_results[best_r_idx]["gamma_train"]):.3f}')

print(f'\n  gamma_geo values across checkpoints:')
for r in geodesic_results:
    print(f'    gamma_train={r["gamma_train"]:.3f}  ->  gamma_geo={r["gamma_geo"]:.4f}')

g_geo_values = [r['gamma_geo'] for r in geodesic_results]
g_geo_std = np.std(g_geo_values)
g_geo_mean = np.mean(g_geo_values)
print(f'\n  gamma_geo mean={g_geo_mean:.4f}  std={g_geo_std:.4f}')
if g_geo_std < 0.05:
    print(f'  >>> gamma_geo converges regardless of training damping.')
    print(f'      Intrinsic preferred geometry at gamma ~{g_geo_mean:.3f}.')

print(f'\n  Output files:')
print(f'    Sweep summary:    {summary_path}')
print(f'    Geodesic results: {geo_json_path}')
print(f'    Overlay figure:   {overlay_path}')
print(f'    Gamma_geo figure: {geo_recovery_path}')
print(f'    Per-layer figure: {perlayer_path}')
print(f'\nDone.')